# Can small models match GPT-4o-mini / GPT-5-mini?

A reproducible notebook that tests — on **this repo's own benchmark data** — whether an SLM-only
guard (single model, ensemble, cascade, or learned router) can reach frontier judge quality at a
fraction of the latency/cost.

Plan & rationale: [`docs/slm-parity-plan.md`](../docs/slm-parity-plan.md). This notebook is the
plan's **first concrete step**: reproduce the gap, measure the achievable ceiling, and run the first
learned combiner — all against the cached per-sample predictions already in `outputs/predictions/`.

> Run from the repo root with the project venv. Cells 1–3 need only the repo + numpy (no retraining,
> no API calls). Experiments B–E (calibration, retraining, distillation) are described at the end
> with exact commands.


## 1 · Setup & the current gap


In [ ]:
import json, numpy as np
from agent_bouncer.evaluation.ensembles import load_predictions, _align_rows, diversity_report

blob = json.load(open('outputs/benchmark_results.json'))
results = blob['results']
preds = load_predictions('outputs/predictions')

def cat(g):
    return 'ensemble' if g.startswith('ensemble-') else ('gpt' if g.startswith('openai-') else 'small')

def macro(g):
    ms = [results[b][g] for b in results if g in results[b]]
    keys = ['precision','recall','f1','roc_auc','fpr_on_benign','latency_p50_ms','n']
    out = {}
    for k in keys:
        vals = [m[k] for m in ms if m.get(k) is not None]
        out[k] = sum(vals)/len(vals) if vals else None
    return out

guards = sorted({g for b in results for g in results[b]}, key=lambda g:(cat(g), -(macro(g)['f1'] or 0)))
print(f"{'guard':30s} {'F1':>6s} {'AUC':>6s} {'FPR':>6s} {'p50':>7s} {'n':>5s}")
for g in guards:
    m = macro(g); fmt = lambda x,d=3: f'{x:.{d}f}' if x is not None else '  -  '
    print(f"{g:30s} {fmt(m['f1'])} {fmt(m['roc_auc'])} {fmt(m['fpr_on_benign'])} "
          f"{fmt(m['latency_p50_ms'],0):>7s} {fmt(m['n'],0):>5s}  [{cat(g)}]")

print('\n⚠ GPT baselines and small models were scored at different per-class this run '
      '(see the n column) — re-score at matched n before any parity claim (plan §4).')

## 2 · The ceiling — are the members complementary?

The **oracle ceiling** is the accuracy you'd reach if you could route each sample to the member that
gets it right. If it's far above the best single model *and* errors are uncorrelated, the information
to beat the frontier is already in the pool — the bottleneck is the **combiner**, not capacity.


In [ ]:
pool = [n for n in preds if cat(n) == 'small' and sum(len(v) for v in preds[n].values()) > 50]
rep = diversity_report(preds, pool)
print(f"members            : {len(pool) - len(rep['dropped'])}  (dropped: {rep['dropped']})")
print(f"best single accuracy: {rep['best_single_accuracy']:.3f}")
print(f"ORACLE ceiling      : {rep['oracle_accuracy']:.3f}   <- route-to-the-right-member upper bound")
print(f"headroom            : {rep['headroom']:.3f}")
print(f"error-correlation Q : {rep['mean_error_correlation']:.3f}  (low = complementary)")
print(f"verdict             : {rep['verdict'].upper()} — {rep['explanation']}")

## 3 · Experiment A — a learned router (stacking) over the members

Can a small meta-model *capture* that headroom? We fit a logistic-regression router on the members'
per-sample scores, evaluated **leave-one-benchmark-out** (grouped CV) so it must *generalize* across
benchmarks — the honest test, no cross-benchmark leakage.


In [ ]:
benches = set.intersection(*[set(preds[m]) for m in pool])
X, y, groups = [], [], []
for bi, b in enumerate(sorted(benches)):
    rows = _align_rows([preds[m][b] for m in pool])
    if rows is None:
        continue
    for i in range(len(rows[0])):
        X.append([rows[mi][i][2] for mi in range(len(pool))])  # each member's score
        y.append(rows[0][i][0]); groups.append(bi)
X, y, groups = np.array(X, float), np.array(y, float), np.array(groups)

def f1(yt, yp):
    tp = ((yp==1)&(yt==1)).sum(); fp = ((yp==1)&(yt==0)).sum(); fn = ((yp==0)&(yt==1)).sum()
    p = tp/(tp+fp) if tp+fp else 0; r = tp/(tp+fn) if tp+fn else 0
    return 2*p*r/(p+r) if p+r else 0.0

def fit(Xtr, ytr, epochs=400, lr=0.5):
    Xb = np.c_[np.ones(len(Xtr)), Xtr]; w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        p = 1/(1+np.exp(-Xb@w)); w -= lr * Xb.T@(p-ytr)/len(Xb)
    return w
predict = lambda w, Xte: (1/(1+np.exp(-np.c_[np.ones(len(Xte)), Xte]@w)) >= 0.5).astype(float)

cv = [f1(y[groups==b], predict(fit(X[groups!=b], y[groups!=b]), X[groups==b])) for b in np.unique(groups)]
best_single = max(f1(y, X[:, i] >= 0.5) for i in range(X.shape[1]))
print(f'samples={len(y)}  members={X.shape[1]}')
print(f'best single (pooled) macro-F1 : {best_single:.3f}')
print(f'stacked router (grouped CV)   : {np.mean(cv):.3f}')
print(f'oracle ceiling                : {rep["oracle_accuracy"]:.3f}')

### Reading Experiment A

Expect the router to land **near or below best-single**, well short of the oracle. That's the key
insight, not a failure: the member scores are **binary (0/1)**, so a linear router has almost nothing
to learn from, and reliability varies per benchmark so it doesn't transfer. **The headroom is real but
not reachable with binary votes** — which is exactly why the plan orders the next levers as:

1. **L3 — calibrated continuous scores** (temperature/Platt on a logit or `unsafe`-token logprob) so
   the router/soft-vote/deferral have real signal;
2. **L4 — frontier distillation** (label a pool with GPT-4o-mini, SFT a small student) to move the
   base decision boundary toward the teacher;
3. **L1/L5 — fix the degenerate RL/DPO members and add red-team data** to make the pool both usable
   and diverse.


## 4 · Next experiments (B–E) — commands

```bash
# §4 (do first): re-score the frontier judges at the SAME per-class as the SLMs (fair comparison)
python scripts/eval/run_benchmarks.py --per-class 0 --skip-decoder \
    --guards openai-moderation openai-gpt-4o-mini openai-gpt-5.4-mini

# L1 — fix a degenerate RL/DPO member (stronger false-positive penalty), then re-test
python scripts/train/run_training.py --model qwen3-1.7b --technique dpo --beta 0.3 ...

# L5 — close the red-team gap: build injection/jailbreak-heavy training data, retrain SFT
python scripts/data/build_dataset.py --strategy red_team --name redteam \
    --sources prompt_injections jailbreak_classification jailbreakbench
python scripts/train/run_training.py --model qwen3-1.7b --technique sft --train-data data/train_sets/redteam/train.jsonl

# L4 — distill GPT-4o-mini into a fast student (label a pool with the teacher, then SFT)
#   see docs/slm-parity-plan.md §3 cell 7
```

**Success criteria (plan §5):** an SLM-only pipeline that, at matched n, has macro-F1 ≥ GPT-5-mini
with FPR@benign ≤ GPT-5-mini, is within 0.03 F1 of GPT-4o-mini with FPR ≤ its, and runs at ≤ 1/3 the
frontier latency with no per-request API cost. Stretch: a **single distilled SLM** that clears the
GPT-5-mini bar.
